# 01 — Session-grouped cross-validation: detector training and out-of-fold predictions

**Visual–thermal monkey detection across times of day** · [Repository](https://github.com/ellail-mardi/monkey-detection)

This notebook produces the **out-of-fold (OOF) predictions** used by `02_evaluation.ipynb`.

**Protocol**
1. The 120 test pairs are grouped by recording **session** (frames of one session are highly correlated, so random frame-level splits would be optimistic) and partitioned into *K* = 3 folds that are balanced in size and across the four time windows.
2. For each fold *k*, a visual and a thermal **YOLO11n** detector are trained from COCO-pretrained weights on *the original training pairs + the test sessions of the other folds*, using the training recipe stored in the original checkpoints (`weights/*.pt`) and a fixed number of epochs. The last-epoch checkpoint is used, so no model selection is made on held-out data.
3. Each held-out fold is predicted by the detectors that never saw its sessions; the three predictions are pooled into `cv/oof_preds.pkl`.

Both detectors are single-class ("monkey"), input size 575 px (rounded to 576 by Ultralytics).

**Run:** Runtime → Change runtime type → GPU, then Run all (6 trainings, roughly 5–10 min each on a T4). Finished folds are skipped, so the notebook can be resumed after a disconnect (set `USE_GOOGLE_DRIVE = True` to keep files across sessions). Set `QUICK_TEST = True` for a 2-epoch dry run that writes to `cv/quick/` only.

## 1. Install

In [ ]:
!pip -q install -U ultralytics

## 2. Configuration and data discovery

In [ ]:
import os, re, io, sys, json, time, shutil, hashlib, pickle, random, warnings, platform, zipfile, subprocess, itertools
from pathlib import Path
from collections import OrderedDict, Counter, defaultdict
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# ----------------------------- Locations -----------------------------
REPO_URL = "https://github.com/ellail-mardi/monkey-detection"
REPO_DIR = Path(os.environ.get("ALF_REPO", "/content/monkey-detection"))   # where the repository is (or will be cloned)
USE_GOOGLE_DRIVE = False        # True: write generated files (cv/, results/) to Google Drive so they survive a Colab disconnect
DRIVE_OUT = "/content/drive/MyDrive/monkey-detection-outputs"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive"); OUTPUT_ROOT = Path(DRIVE_OUT)
else:
    OUTPUT_ROOT = REPO_DIR
CV_DIR, RESULTS_DIR = OUTPUT_ROOT / "cv", OUTPUT_ROOT / "results"
CV_DIR.mkdir(parents=True, exist_ok=True); RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------- Data layout (auto-detected) -----------------------------
# Expected layout (folder names are matched case-insensitively; Malay names are accepted as aliases):
#   <data root>/test/{dawn|subuh, midday|tengahari, dusk|senja, night|malam}/{visual, thermal, labels}
#   <training folder>/val/{visual, thermal, labels}
#   weights/visual_best.pt, weights/thermal_best.pt  (source-only detectors, optional)
# Override any of the following with an explicit Path if auto-detection fails.
DATA_ROOT = None; TRAIN_DIR = None; TEST_DIR = None; W_VIS = None; W_THR = None

COND_ALIASES = OrderedDict([("dawn", ["dawn", "subuh"]), ("midday", ["midday", "tengahari"]),
                            ("dusk", ["dusk", "senja"]), ("night", ["night", "malam"])])
COND_MAP = OrderedDict([("dawn", "Dawn"), ("midday", "Midday"), ("dusk", "Dusk"), ("night", "Night")])
def _child(d, names):
    if d is None or not Path(d).exists(): return None
    low = {p.name.lower(): p for p in Path(d).iterdir() if p.is_dir()}
    for n in names:
        if n.lower() in low: return low[n.lower()]
    return None
def _walk_dirs(root, max_depth):
    queue = [(Path(root), 0)]
    while queue:
        d, k = queue.pop(0); yield d
        if k < max_depth:
            queue += [(p, k + 1) for p in sorted(d.iterdir()) if p.is_dir() and not p.name.startswith(".")]
def locate():
    global DATA_ROOT, TRAIN_DIR, TEST_DIR, W_VIS, W_THR
    if TEST_DIR is None:
        if DATA_ROOT is None:
            for d in _walk_dirs(REPO_DIR, 3):
                t = _child(d, ["test"])
                if t is not None and all(_child(t, a) is not None for a in COND_ALIASES.values()): DATA_ROOT = d; break
        assert DATA_ROOT is not None, "Could not find a 'test' folder with the four time-of-day sub-folders. Set DATA_ROOT / TEST_DIR manually."
        TEST_DIR = _child(DATA_ROOT, ["test"])
    if TRAIN_DIR is None:
        base = DATA_ROOT if DATA_ROOT is not None else TEST_DIR.parent
        for d in _walk_dirs(base, 2):
            if d == TEST_DIR or TEST_DIR in d.parents: continue
            if _child(d, ["visual"]) is not None and _child(d, ["thermal"]) is not None: TRAIN_DIR = d; break
    if W_VIS is None: W_VIS = next(iter(sorted(REPO_DIR.rglob("visual_best.pt"))), None)
    if W_THR is None: W_THR = next(iter(sorted(REPO_DIR.rglob("thermal_best.pt"))), None)
locate()
print("Repository      :", REPO_DIR); print("Test folder     :", TEST_DIR); print("Training folder :", TRAIN_DIR)
print("Source weights  :", W_VIS, "|", W_THR); print("Outputs         :", CV_DIR, "|", RESULTS_DIR)

K = 3
SEED = 42
IMGSZ = 575
CONF_PRED, NMS_IOU, MAX_DET, BATCH_PRED = 0.001, 0.6, 100, 16
BASE_FALLBACK = "yolo11n.pt"            # used if the base model cannot be read from the original checkpoints
EPOCHS_OVERRIDE, BATCH_OVERRIDE = None, None   # None = use the original training recipe
QUICK_TEST = False                       # True: 2 epochs, outputs go to cv/quick (real outputs are never touched)

random.seed(SEED); np.random.seed(SEED)
CVW = CV_DIR / "quick" if QUICK_TEST else CV_DIR
CVW.mkdir(parents=True, exist_ok=True)
WORK = Path(os.environ.get("ALF_WORK", "/content/cv_data"))      # local copy of the datasets built for each fold
RUNS = Path(os.environ.get("ALF_RUNS", "/content/cv_runs"))
assert TRAIN_DIR is not None, "Training folder not found: set TRAIN_DIR manually in the cell above."

## 3. Collect frame pairs and annotations (single class)

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
DUP_KEYS = []
def _pair_key(p):
    # pair key = session_frame, e.g. v19_00001.png / t19_00001.png -> "19_00001" (robust to Roboflow-style suffixes)
    m = re.search(r"(\d+_\d+)", p.stem)
    return m.group(1) if m else re.sub(r"^[A-Za-z]+[_\-]?", "", p.stem)
def _list_images(d):
    out = {}
    if d is None or not Path(d).exists(): return out
    for p in sorted(Path(d).iterdir()):
        if p.suffix.lower() not in IMG_EXT: continue
        k = _pair_key(p)
        if k in out: DUP_KEYS.append(f"{d}: {out[k].name} vs {p.name}")
        out[k] = p
    return out
def session_of(p):
    m = re.match(r"[A-Za-z]*(\d+)_", p.stem); return m.group(1) if m else "?"
def load_boxes(p):
    """YOLO label file -> normalised xyxy boxes (polygons are converted to their bounding box)."""
    if p is None or not Path(p).exists(): return np.zeros((0, 4), np.float32)
    B = []
    for ln in Path(p).read_text().strip().splitlines():
        t = ln.split()
        if len(t) == 5:
            x, y, w, h = map(float, t[1:]); B.append([x - w/2, y - h/2, x + w/2, y + h/2])
        elif len(t) > 5:
            xs, ys = np.array(t[1::2], float), np.array(t[2::2], float); B.append([xs.min(), ys.min(), xs.max(), ys.max()])
    return np.clip(np.array(B, np.float32).reshape(-1, 4), 0, 1)
def _iou1(a, b):
    iw, ih = min(a[2], b[2]) - max(a[0], b[0]), min(a[3], b[3]) - max(a[1], b[1]); inter = max(iw, 0) * max(ih, 0)
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-9)
GT_DEDUP_IOU = 0.95        # after merging all classes into one, drop duplicate boxes (e.g. a visual and a thermal box on the same animal)
def dedup(b):
    keep = []
    for k in range(len(b)):
        if all(_iou1(b[k], b[j]) < GT_DEDUP_IOU for j in keep): keep.append(k)
    return b[keep]
def collect_pairs(d, cond, split):
    """Pair visual/thermal frames of a folder by session_frame key; attach (single-class) ground-truth boxes."""
    d = Path(d); vis, thr = _list_images(_child(d, ["visual"])), _list_images(_child(d, ["thermal"])); lab = _child(d, ["labels"])
    recs = []
    for k in sorted(set(vis) & set(thr)):
        lv = lab / f"{vis[k].stem}.txt" if lab else None; lt = lab / f"{thr[k].stem}.txt" if lab else None
        lv_ok, lt_ok = bool(lv and lv.exists()), bool(lt and lt.exists())
        label_v = lv if lv_ok else (lt if lt_ok else None)
        recs.append(dict(split=split, cond=cond, key=k, vis=vis[k], thr=thr[k], session=session_of(vis[k]),
                         boxes_V=dedup(load_boxes(label_v)),                       # boxes for the visual frame
                         boxes_T=dedup(load_boxes(lt if lt_ok else label_v)),      # boxes for the thermal frame (own file if present)
                         has_thermal_label_file=lt_ok and lt != label_v))
    return recs, sorted(set(vis) - set(thr)), sorted(set(thr) - set(vis))
def collect_test():
    recs, problems = [], []
    for c, aliases in COND_ALIASES.items():
        r, ov, ot = collect_pairs(_child(TEST_DIR, aliases), c, "test"); recs += r
        if ov or ot: problems.append(f"[test/{c}] unpaired frames -> visual only: {ov} | thermal only: {ot}")
    return recs, problems

pool, ov, ot = collect_pairs(TRAIN_DIR, "train", "train")
test, problems = collect_test()
print(f"Training pairs: {len(pool)} | separate thermal label files: {sum(r['has_thermal_label_file'] for r in pool)}/{len(pool)}")
print(f"Test pairs    : {len(test)} | per window: {dict(Counter(COND_MAP[r['cond']] for r in test))}")
for p in problems + ([f"[train] unpaired frames: visual-only {ov}, thermal-only {ot}"] if (ov or ot) else []): print("WARNING:", p)
if DUP_KEYS: print("WARNING: duplicate pair keys:", DUP_KEYS[:3])
assert len(pool) > 0 and len(test) > 0, "No data found: check the folder layout."
if not all(r["has_thermal_label_file"] for r in pool):
    print("WARNING: some training pairs have no separate thermal label file; the visual label is used for the thermal detector (the two cameras are not co-located, so this may be misaligned).")
print("Instances: training (visual labels)", sum(len(r["boxes_V"]) for r in pool), "| training (thermal labels)", sum(len(r["boxes_T"]) for r in pool), "| test", sum(len(r["boxes_V"]) for r in test))

## 4. Partition test sessions into balanced folds

In [ ]:
sinfo = {}
for r in test:
    d = sinfo.setdefault(r["session"], dict(cond=r["cond"], n=0)); d["n"] += 1
    assert d["cond"] == r["cond"], f"Session {r['session']} occurs in more than one time window"
skeys = sorted(sinfo); conds = list(COND_MAP); ncond = {c: sum(v["n"] for v in sinfo.values() if v["cond"] == c) for c in conds}
size = np.array([sinfo[s]["n"] for s in skeys]); cidx = np.array([conds.index(sinfo[s]["cond"]) for s in skeys])

def penalty(a):                      # squared deviation from equal fold sizes and from equal per-window shares
    a = np.asarray(a); p = 0.0
    for f in range(K):
        m = a == f; p += (size[m].sum() - len(test) / K) ** 2
        for ci, c in enumerate(conds): p += (size[m & (cidx == ci)].sum() - ncond[c] / K) ** 2
    return p
if len(skeys) <= 14:
    best = min(itertools.product(range(K), repeat=len(skeys)), key=penalty)            # exhaustive, deterministic
else:
    rng_ = np.random.default_rng(SEED); best = min((rng_.integers(0, K, len(skeys)) for _ in range(200000)), key=penalty)
FOLD_OF = {s: int(f) for s, f in zip(skeys, best)}
tab = defaultdict(Counter)
for r in test: tab[FOLD_OF[r["session"]]][COND_MAP[r["cond"]]] += 1
print("Fold -> sessions:", {f: [s for s in skeys if FOLD_OF[s] == f] for f in range(K)})
for f in range(K): print(f"  fold {f}: {sum(tab[f].values())} pairs", dict(tab[f]))
train_sess, test_sess = {session_of(r["vis"]) for r in pool}, set(skeys)
if train_sess & test_sess:
    print(f"\nNOTE: session IDs {sorted(train_sess & test_sess)} occur in both training and test folders. "
          "This is fine if numbering restarts per folder (different recordings); if they are the same videos, those sessions are not unseen.")
json.dump(dict(K=K, fold_of=FOLD_OF, sessions=sinfo, seed=SEED), open(CVW / "folds.json", "w"), indent=1)

## 5. Training recipe and training functions

In [ ]:
import torch
from ultralytics import YOLO
from importlib.metadata import version as _v
_uv = tuple(int(x) for x in re.findall(r"\d+", _v("ultralytics"))[:2])
assert _uv >= (8, 3), f"ultralytics {_v('ultralytics')} is too old for YOLO11 (need >= 8.3): pip install -U ultralytics, then restart the session"
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("ultralytics", _v("ultralytics"), "| device:", torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU (slow; a GPU is recommended)")
try:
    from ultralytics.cfg import DEFAULT_CFG_DICT; VALID = set(DEFAULT_CFG_DICT)
except Exception:
    VALID = None
KEEP = ["epochs", "batch", "lr0", "lrf", "momentum", "weight_decay", "optimizer", "warmup_epochs", "warmup_momentum", "warmup_bias_lr",
        "hsv_h", "hsv_s", "hsv_v", "degrees", "translate", "scale", "shear", "perspective", "flipud", "fliplr", "bgr", "mosaic", "mixup", "cutmix",
        "copy_paste", "close_mosaic", "cos_lr", "dropout", "label_smoothing", "box", "cls", "dfl", "nbs", "multi_scale", "rect", "freeze"]
W_ORIG = {"visual": W_VIS, "thermal": W_THR}
def read_args(p):
    if p is None: return {}
    try: return torch.load(str(p), map_location="cpu", weights_only=False).get("train_args", {}) or {}
    except Exception as e: print("WARNING: cannot read train_args from", Path(p).name, "-", e); return {}
def recipe_for(mod):
    a = read_args(W_ORIG[mod]); out = {k: a[k] for k in KEEP if k in a and a[k] is not None and (VALID is None or k in VALID)}
    out["imgsz"] = IMGSZ; out.setdefault("epochs", 100); out.setdefault("batch", 16)
    if EPOCHS_OVERRIDE: out["epochs"] = EPOCHS_OVERRIDE
    if BATCH_OVERRIDE: out["batch"] = BATCH_OVERRIDE
    if QUICK_TEST: out["epochs"] = 2
    base = Path(str(a.get("model", ""))).name
    if not (base.startswith("yolo") and base.endswith(".pt")): print(f"INFO: base model for {mod} not recognised ('{base}'); using {BASE_FALLBACK}"); base = BASE_FALLBACK
    return out, base
RECIPE = {m: recipe_for(m) for m in ("visual", "thermal")}
for m, (a, b) in RECIPE.items(): print(m, "| base:", b, "|", a)

def write_label(path, boxes):
    path.write_text("\n".join(f"0 {(b[0]+b[2])/2:.6f} {(b[1]+b[3])/2:.6f} {b[2]-b[0]:.6f} {b[3]-b[1]:.6f}" for b in boxes))

def build_dataset(k, mod):
    """Training data for fold k: all training pairs + test pairs of the other folds (modality-specific images and boxes)."""
    root = WORK / f"fold{k}" / mod; shutil.rmtree(root, ignore_errors=True)
    for sp in ("train", "val"): (root / sp / "images").mkdir(parents=True); (root / sp / "labels").mkdir(parents=True)
    img_key, box_key = ("vis", "boxes_V") if mod == "visual" else ("thr", "boxes_T"); items = []
    for r in pool: items.append((f"train_{r[img_key].name}", r[img_key], r[box_key]))
    for r in test:
        if FOLD_OF[r["session"]] != k: items.append((f"test_{r['cond']}_{r[img_key].name}", r[img_key], r["boxes_V"]))   # test pairs share one set of boxes
    for n, (name, src, boxes) in enumerate(items):
        shutil.copy(src, root / "train" / "images" / name); write_label(root / "train" / "labels" / (Path(name).stem + ".txt"), boxes)
        if n < 20: shutil.copy(src, root / "val" / "images" / name); write_label(root / "val" / "labels" / (Path(name).stem + ".txt"), boxes)   # placeholder validation split (val=False: never used for selection)
    (root / "data.yaml").write_text(f"path: {root}\ntrain: train/images\nval: val/images\nnames:\n  0: monkey\n")
    return root / "data.yaml", len(items)

TRAIN_LOG = {}
def train_fold(k, mod):
    dst = CVW / f"fold{k}" / f"{mod}_last.pt"; dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists(): print(f"[fold {k} / {mod}] already trained -> skipped"); return dst
    data, n = build_dataset(k, mod); a, base = RECIPE[mod]; t0 = time.time()
    print(f"[fold {k} / {mod}] training on {n} images, base={base}, epochs={a['epochs']}")
    model = YOLO(base)
    model.train(data=str(data), project=str(RUNS), name=f"fold{k}_{mod}", exist_ok=True, seed=SEED, deterministic=True, val=False, plots=False,
                patience=int(a["epochs"]) + 1, device=DEVICE, workers=2, verbose=False, **a)
    last = Path(getattr(model.trainer, "last", RUNS / f"fold{k}_{mod}" / "weights" / "last.pt"))
    shutil.copy(last, dst); TRAIN_LOG[f"fold{k}_{mod}"] = dict(n_train_images=n, minutes=round((time.time() - t0) / 60, 1), recipe=a, base=base)
    print(f"   done in {(time.time()-t0)/60:.1f} min -> {dst}")
    return dst

## 6. Train 3 folds × 2 modalities (resumable)

In [ ]:
for k in range(K):
    for mod in ("visual", "thermal"): train_fold(k, mod)
json.dump(TRAIN_LOG, open(CVW / "train_log.json", "w"), indent=1, default=str)
print("\nAll fold models are ready.")

## 7. Out-of-fold predictions for every test pair

In [ ]:
def predict_paths(model, paths):
    out = []
    for i in range(0, len(paths), BATCH_PRED):
        res = model.predict([str(p) for p in paths[i:i + BATCH_PRED]], imgsz=IMGSZ, conf=CONF_PRED, iou=NMS_IOU, max_det=MAX_DET, device=DEVICE, verbose=False)
        for r in res:
            b = r.boxes; cf = b.conf.cpu().numpy().astype(np.float32)
            out.append((b.xyxyn.cpu().numpy().astype(np.float32), cf, np.zeros(len(cf), int)))      # single class
    return out
OOF = {}
for k in range(K):
    held = [r for r in test if FOLD_OF[r["session"]] == k]
    mv = YOLO(str(CVW / f"fold{k}" / "visual_last.pt")); mt = YOLO(str(CVW / f"fold{k}" / "thermal_last.pt"))
    pv, pt = predict_paths(mv, [r["vis"] for r in held]), predict_paths(mt, [r["thr"] for r in held])
    for r, a, b in zip(held, pv, pt): OOF[f'{r["cond"]}/{r["vis"].name}'] = dict(V=a, T=b, fold=k, session=r["session"])
    print(f"fold {k}: {len(held)} held-out pairs predicted")
assert len(OOF) == len(test), "OOF size does not match the number of test pairs"
pickle.dump(OOF, open(CVW / "oof_preds.pkl", "wb"))
print("Saved:", CVW / "oof_preds.pkl")
print("\nNext: run 02_evaluation.ipynb (it picks up cv/oof_preds.pkl automatically).")
if QUICK_TEST: print("QUICK_TEST: outputs are in cv/quick only; set QUICK_TEST = False for the real run.")